# 🎙️ KuroYs AutoMatch — Tạo voice bằng Google Colab

Dành cho máy **không có GPU NVIDIA** (hoặc GPU yếu): phần tạo voice chạy trên máy chủ của Google.

**Cách dùng**
1. Trong tool, tab **Text to Speech**: chọn giọng, dán kịch bản, bấm **“Tạo voice bằng Colab”** → tool lưu file `colab_voice_….zip`.
2. Ở đây: menu **Runtime → Change runtime type → T4 GPU** → **Save** (chỉ cần làm lần đầu).
3. Menu **Runtime → Run all** (hoặc Ctrl+F9). Khi ô số 4 hiện nút **Choose Files**, chọn file `colab_voice_….zip`.
4. Chờ tạo voice. Xong, trình duyệt tự tải về `colab_result_….zip`.
5. Trong tool bấm **“Nhập kết quả Colab”** → chọn file vừa tải → voice vào dự án, trang Subtitle tự điền.

**Tạo tiếp (cùng dự án hay dự án khác):** không cần tắt Colab — trong tool tạo gói mới, rồi ở đây **chỉ chạy lại
ô 4, 5, 6** (model đã nạp sẵn nên tạo ngay). Chỉ khi Colab báo mất kết nối mới phải **Run all** lại.

**Lưu ý**
- Colab miễn phí có lúc **chưa cấp GPU** hoặc hết giờ dùng trong ngày → thử lại sau (hoặc dùng Colab Pro).
- Bỏ trống khoảng 90 phút Colab tự ngắt → mở lại và **Run all**.
- Giọng mẫu và kịch bản được tải lên máy chủ Google.
- Không cần sửa gì trong các ô bên dưới, chỉ bấm chạy.

In [ ]:
#@title 1. Kiểm tra GPU
import subprocess
try:
    name = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    name = ""
if name:
    print("✅ Đã có GPU:", name)
else:
    print("⚠️ CHƯA CÓ GPU. Vào Runtime → Change runtime type → chọn T4 GPU → Save, rồi Runtime → Run all.")
    print("   Nếu Colab báo hết GPU: thử lại sau ít phút/giờ. Chạy bằng CPU vẫn được nhưng RẤT chậm.")

In [ ]:
#@title 2. Cài đặt (khoảng 1–3 phút, chỉ chạy một lần mỗi phiên)
import os, sys, urllib.request
!pip -q install omnivoice==0.2.1 soundfile==0.14.0
os.makedirs("/content/kuroys", exist_ok=True)
for name in ("kuroys_colab_voice.py", "voice_cleanup.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Kuroys2312/AutoMatchStudio-Releases/colab-v2/colab/" + name, "/content/kuroys/" + name)
sys.path.insert(0, "/content/kuroys")
import kuroys_colab_voice as kcv
CACHE = "/content/kuroys_models"  # ô 3 đổi sang Google Drive nếu bạn tích ô
print("✅ Đã cài xong.")

In [ ]:
#@title 3. (Tùy chọn) Lưu model vào Google Drive — tích ô rồi chạy; không cần thì cứ chạy qua
luu_model_vao_drive = False #@param {type:"boolean"}
if "kcv" not in globals():
    raise SystemExit("❌ Chưa cài đặt. Hãy chạy ô 2 trước (hoặc Runtime → Run all).")
if luu_model_vao_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/KuroYs_Colab/models"
else:
    CACHE = "/content/kuroys_models"
print("Thư mục model:", CACHE)

In [ ]:
#@title 4. Tải gói colab_voice_….zip lên
if "kcv" not in globals():
    raise SystemExit("❌ Chưa cài đặt. Hãy chạy ô 2 trước (hoặc Runtime → Run all).")
from google.colab import files
uploaded = files.upload()
packages = [name for name in uploaded if name.lower().endswith(".zip")]
if not packages:
    raise SystemExit("❌ Chưa chọn file .zip. Chạy lại ô này và chọn file colab_voice_….zip do tool tạo.")
PACKAGE = "/content/" + packages[-1]
job = kcv.load_package(PACKAGE, "/content/kuroys_check")
voices = ", ".join(str(name) for name in (job.get("voices") or {"": job.get("voice_name")}).values())
print(f"✅ Gói: dự án “{job['project_name']}”, giọng {voices}, {len(job['parts'])} phần.")

In [ ]:
#@title 5. Tạo voice
if "kcv" not in globals():
    raise SystemExit("❌ Chưa cài đặt. Hãy chạy ô 2 trước (hoặc Runtime → Run all).")
if "PACKAGE" not in globals():
    raise SystemExit("❌ Chưa tải gói lên. Hãy chạy ô 4 và chọn file colab_voice_….zip.")
CACHE = globals().get("CACHE", "/content/kuroys_models")  # bỏ qua ô 3 vẫn chạy được
if "MODEL" not in globals():
    MODEL = kcv.load_model(CACHE)
RESULT = kcv.run(PACKAGE, "/content/kuroys_work", model_bundle=MODEL)

In [ ]:
#@title 6. Tải kết quả về máy
if "RESULT" not in globals():
    raise SystemExit("❌ Chưa có kết quả. Hãy chạy ô 5 (Tạo voice) trước.")
from google.colab import files
print("Đang tải về:", RESULT)
files.download(RESULT)
print("✅ Xong. Trong tool bấm “Nhập kết quả Colab” và chọn file vừa tải.")